## 02 — Data Preprocessing


In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import math
import glob
import shutil

# ── HDF5 reader (Million Song Dataset files) ──────────────────────────────────
import h5py

# ── PySpark core ──────────────────────────────────────────────────────────────
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, lower, trim, concat_ws
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [2]:
# The 8 audio features we use across all Spotify datasets.
# loudness was dropped — it's on a different scale (-60 to 0 dB) and
# introduces scaling noise when combined with the 0-1 range features.
AUDIO_FEATURES = [
    "danceability", "energy", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

# Columns we keep from each dataset alongside the audio features
METADATA_COLS = ["track_id", "track_name", "artist_name", "genre", "popularity"]

# Output directories
PROCESSED_DIR = "../data/processed"
SPLIT_DIR     = "../outputs/train_test_split"

In [3]:
import sys

# On Windows, the system-level SPARK_HOME may point to a different Spark version.
# We override it here to make sure PySpark always finds its own bundled JARs.
os.environ["SPARK_HOME"]  = os.path.dirname(pyspark.__file__)

# Spark needs Hadoop's winutils.exe on Windows to handle local file operations
# (directory creation, temp files, etc.). We installed winutils at C:\hadoop\bin\.
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Prepend C:\hadoop\bin to PATH so the JVM can find hadoop.dll via
# System.loadLibrary("hadoop"). HADOOP_HOME alone only locates winutils.exe
# for Hadoop's shell helpers — it does NOT add the directory to
# java.library.path. Without this, NativeIO$Windows.access0 throws
# UnsatisfiedLinkError on Spark write operations.
hadoop_bin = r"C:\hadoop\bin"
if hadoop_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")

# Pin the Python interpreter for both the driver and the executor-side workers
# to the exact Python running this notebook. Without these, PySpark on Windows
# can pick a different Python for workers, causing them to die with
# "Connection reset" the moment Spark tries to deserialize Python rows.
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("SPARK_HOME    :", os.environ["SPARK_HOME"])
print("HADOOP_HOME   :", os.environ["HADOOP_HOME"])
print("PATH[0]       :", os.environ["PATH"].split(os.pathsep)[0])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])

SPARK_HOME    : c:\Users\Mariam\anaconda3\envs\spotify-recommender\Lib\site-packages\pyspark
HADOOP_HOME   : C:\hadoop
PATH[0]       : C:\hadoop\bin
PYSPARK_PYTHON: c:\Users\Mariam\anaconda3\envs\spotify-recommender\python.exe


In [4]:
# local[*]  — uses all CPU cores on this machine (pseudo-distributed mode).
# spark.driver.memory — heap given to the JVM driver process.
# spark.local.dir     — MUST be on the same drive as the output directory (D:).
#   Spark writes temp files here, then renames them to the final output path.
#   Java's File.renameTo() cannot rename across different drives on Windows,
#   so if temp is on C: and output is on D:, every write fails silently.
# spark.hadoop.home.dir — tells the JVM where winutils lives at runtime.
# RawLocalFileSystem  — skips Unix chmod/chown calls that crash on Windows.
# fileoutputcommitter.algorithm.version=2 — avoids the rename-based commit phase.
# marksuccessfuljobs=false — skips writing the _SUCCESS marker file.

SPARK_TEMP_DIR = "D:/tmp/spark"
os.makedirs(SPARK_TEMP_DIR, exist_ok=True)

spark = SparkSession.builder \
    .appName("SpotifyPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.local.dir", SPARK_TEMP_DIR) \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


def spark_write_csv(df, relative_path):
    """
    Write a Spark DataFrame to a single CSV file on Windows.
    Deletes the output directory with Python first so Spark never has to
    rename or overwrite an existing folder across drives.
    """
    if os.path.exists(relative_path):
        shutil.rmtree(relative_path)
    abs_path = os.path.abspath(relative_path).replace("\\", "/")
    df.coalesce(1).write.csv(f"file:///{abs_path}", header=True)
    print(f"Saved → {relative_path}/")

Spark version: 3.5.8


2.1 - Load Data


In [5]:
DATA_DIR = "../data/raw"

# The 5 Kaggle Spotify datasets — each has slightly different column names
# and a different subset of audio features, which we standardize in the next cell.
# spotify_2023 is loaded separately because its column names differ significantly.
datasets = {
    "spotify_tracks":  f"{DATA_DIR}/spotify_tracks/dataset.csv",
    "genres_v2":       f"{DATA_DIR}/dataset_of_songs/genres_v2.csv",
    "audio_apr2019":   f"{DATA_DIR}/spotify_audio_features/SpotifyAudioFeaturesApril2019.csv",
    "audio_nov2018":   f"{DATA_DIR}/spotify_audio_features/SpotifyAudioFeaturesNov2018.csv",
    "ultimate_tracks": f"{DATA_DIR}/ultimate_spotify_tracks/SpotifyFeatures.csv",
}

dfs = {}
for name, path in datasets.items():
    dfs[name] = spark.read.csv(path, header=True, inferSchema=True)
    print(f"{name}: {dfs[name].count():,} rows × {len(dfs[name].columns)} cols")

spotify_tracks: 114,000 rows × 21 cols
genres_v2: 42,305 rows × 22 cols
audio_apr2019: 130,663 rows × 17 cols
audio_nov2018: 116,372 rows × 17 cols
ultimate_tracks: 232,725 rows × 18 cols


2.2 - Standardize Schemas


In [6]:
# Helper function: maps each dataset's unique column names to our standard schema.
# Any column that doesn't exist in a dataset (e.g. genre in audio_apr2019)
# is filled with lit(None) so all 6 DataFrames have identical schemas for union.
def select_standard(df, track_id, track_name, artist_name, genre, popularity):
    return df.select(
        col(track_id).alias("track_id"),
        col(track_name).alias("track_name"),
        col(artist_name).alias("artist_name"),
        col(genre).alias("genre"),
        col(popularity).cast("double").alias("popularity"),
        *[col(f).cast("double") for f in AUDIO_FEATURES]
    )

In [7]:
# df1 — spotify_tracks (114k rows): has track_id, track_name, artists, track_genre, popularity
df1 = select_standard(dfs["spotify_tracks"],
    "track_id", "track_name", "artists", "track_genre", "popularity")

# df2 — genres_v2 (42k rows): no artist or popularity columns → filled with None
df2 = dfs["genres_v2"].select(
    col("id").alias("track_id"),
    col("song_name").alias("track_name"),
    lit(None).cast("string").alias("artist_name"),
    col("genre").alias("genre"),
    lit(None).cast("double").alias("popularity"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# df3 — audio_apr2019 (130k rows): has artist and popularity but no genre
df3 = dfs["audio_apr2019"].select(
    col("track_id"), col("track_name"), col("artist_name"),
    lit(None).cast("string").alias("genre"),
    col("popularity").cast("double"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# df4 — audio_nov2018 (116k rows): same structure as apr2019, no genre
df4 = dfs["audio_nov2018"].select(
    col("track_id"), col("track_name"), col("artist_name"),
    lit(None).cast("string").alias("genre"),
    col("popularity").cast("double"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# df5 — ultimate_tracks (232k rows): most complete dataset, has all columns
df5 = dfs["ultimate_tracks"].select(
    col("track_id"), col("track_name"), col("artist_name"),
    col("genre"), col("popularity").cast("double"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# df6 — spotify_2023 (375k rows): largest dataset, but uses different column names
# artist_0 = first artist, genre_0 = primary genre, track_popularity = popularity score
df_2023_raw = spark.read.csv(
    f"{DATA_DIR}/spotify_2023/spotify_data_12_20_2023.csv",
    header=True, inferSchema=True
)
df6 = df_2023_raw.select(
    col("track_id"), col("track_name"),
    col("artist_0").alias("artist_name"),
    col("genre_0").alias("genre"),
    col("track_popularity").cast("double").alias("popularity"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# MSD is kept separate — it only has 3 real audio features vs our 8
print("Individual dataset sizes:")
for name, d in [("spotify_tracks", df1), ("genres_v2", df2), ("audio_apr2019", df3),
                ("audio_nov2018", df4), ("ultimate_tracks", df5), ("spotify_2023", df6)]:
    print(f"  {name}: {d.count():,} rows")

Individual dataset sizes:
  spotify_tracks: 114,000 rows
  genres_v2: 42,305 rows
  audio_apr2019: 130,663 rows
  audio_nov2018: 116,372 rows
  ultimate_tracks: 232,725 rows
  spotify_2023: 375,141 rows


In [8]:
# unionByName stacks all 6 DataFrames vertically, matching columns by name.
# This is safer than union() which matches by position — important since
# each dataset was built with a slightly different select() order.
songs_df_merged = (df1.unionByName(df2)
                      .unionByName(df3)
                      .unionByName(df4)
                      .unionByName(df5)
                      .unionByName(df6))

total = songs_df_merged.count()
print(f"Total rows after union: {total:,}")

Total rows after union: 1,011,206


In [9]:
# Cache the merged DataFrame so Spark doesn't recompute it from scratch
# every time we call count() or run an action below.
songs_df_merged.cache()

# Step 1: Drop rows where any audio feature is null.
# We allow null metadata (genre, artist) — those won't break the ML models.
songs_df_cleaned = songs_df_merged.dropna(subset=AUDIO_FEATURES)

# Step 2: Remove duplicate track IDs — same song appearing in multiple datasets.
songs_df_cleaned = songs_df_cleaned.dropDuplicates(["track_id"])

# Step 3: Remove songs with the same name + artist but different IDs
# (e.g., a track re-uploaded to Spotify gets a new track_id but is the same song).
songs_df_cleaned = songs_df_cleaned.dropDuplicates(["track_name", "artist_name"])

after = songs_df_cleaned.count()
print(f"Rows after cleaning : {after:,}")
print(f"Rows removed        : {total - after:,}")

Rows after cleaning : 697,392
Rows removed        : 313,814


2.3 - Feature Selection


In [10]:
# Drop all columns except metadata + audio features.
# We don't need things like key, mode, time_signature, etc.
songs_df_finalized = songs_df_cleaned.select(METADATA_COLS + AUDIO_FEATURES)

songs_df_finalized.show(5)
print(f"Final shape: {songs_df_finalized.count():,} rows × {len(songs_df_finalized.columns)} cols")

+--------------------+--------------------+-----------------+-----+----------+------------+------+-----------+------------+----------------+--------+------------------+-------+
|            track_id|          track_name|      artist_name|genre|popularity|danceability|energy|speechiness|acousticness|instrumentalness|liveness|           valence|  tempo|
+--------------------+--------------------+-----------------+-----+----------+------------+------+-----------+------------+----------------+--------+------------------+-------+
|0wtpkz93wATDkUExJ...|""" La Traviata "...|     Maria Callas|Opera|      31.0|       0.364| 0.275|      0.043|       0.993|          0.0284|   0.293|            0.0394| 86.096|
|6YQUuoMnRIMaOmouY...|            """99"""|   Barns Courtney| NULL|      69.0|       0.552| 0.804|     0.0303|     0.00598|             0.0|   0.111|0.7140000000000001|  95.98|
|4oDWA9n9KbHyrM7Cz...|"""A wooden ring"...|  Angela Lansbury|Movie|      11.0|       0.603| 0.122|      0.579|     

2.4 - Save new dataset


In [11]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR,     exist_ok=True)

# Save the cleaned Spotify songs dataset
spark_write_csv(songs_df_finalized, f"{PROCESSED_DIR}/songs_cleaned")

print(f"Total rows : {songs_df_finalized.count():,}")
print(f"Columns    : {songs_df_finalized.columns}")

Saved → ../data/processed/songs_cleaned/
Total rows : 697,392
Columns    : ['track_id', 'track_name', 'artist_name', 'genre', 'popularity', 'danceability', 'energy', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [12]:
# randomSplit is reproducible with seed=42 — same split every run.
songs_train_df, songs_test_df = songs_df_finalized.randomSplit([0.8, 0.2], seed=42)

spark_write_csv(songs_train_df, f"{SPLIT_DIR}/songs_train")
spark_write_csv(songs_test_df,  f"{SPLIT_DIR}/songs_test")

print(f"Train : {songs_train_df.count():,} rows")
print(f"Test  : {songs_test_df.count():,} rows")

Saved → ../outputs/train_test_split/songs_train/
Saved → ../outputs/train_test_split/songs_test/
Train : 558,288 rows
Test  : 139,104 rows


2.5 - Clean MSD Dataset (Remove Spotify Overlaps)


In [13]:
MSD_DIR = f"{DATA_DIR}/million_song/MillionSongSubset"

# MSD only has 3 real audio features. The other 5 Spotify features don't exist in MSD,
# so we work with these 3 only — no fake zeros.
MSD_AUDIO_FEATURES = ["danceability", "energy", "tempo"]

# Explicit schema so Spark doesn't waste time inferring types from 10,000 files
msd_schema = StructType([
    StructField("track_id",     StringType(), True),
    StructField("track_name",   StringType(), True),
    StructField("artist_name",  StringType(), True),
    StructField("genre",        StringType(), True),
    StructField("popularity",   DoubleType(), True),
    StructField("danceability", DoubleType(), True),
    StructField("energy",       DoubleType(), True),
    StructField("tempo",        DoubleType(), True),
])

def read_msd_file(filepath):
    """Read one MSD .h5 file and return a tuple matching msd_schema."""
    with h5py.File(filepath, "r") as f:
        meta     = f["metadata"]["songs"]
        analysis = f["analysis"]["songs"]

        # song_hotttnesss is MSD's popularity proxy (0.0–1.0). Scale to 0–100
        # to match Spotify's popularity column. NaN means no rating available.
        hotttnesss = float(meta["song_hotttnesss"][0])
        popularity = round(hotttnesss * 100, 1) if not math.isnan(hotttnesss) else None

        return (
            meta["song_id"][0].decode("utf-8", errors="ignore"),
            meta["title"][0].decode("utf-8", errors="ignore"),
            meta["artist_name"][0].decode("utf-8", errors="ignore"),
            meta["genre"][0].decode("utf-8", errors="ignore") or None,
            popularity,
            float(analysis["danceability"][0]),
            float(analysis["energy"][0]),
            float(analysis["tempo"][0]),
        )

# Scan all nested subdirectories for .h5 files (MSD uses A/B/C/track.h5 structure)
msd_files = glob.glob(f"{MSD_DIR}/**/*.h5", recursive=True)
print(f"Found {len(msd_files):,} MSD files")

msd_records = []
for fp in msd_files:
    try:
        msd_records.append(read_msd_file(fp))
    except Exception:
        pass  # skip any corrupted files silently

# Build the Spark DataFrame from a SINGLE-partition RDD instead of
# spark.createDataFrame(records, schema). The default createDataFrame splits the
# Python list across local[*] cores, spawning one Python worker per partition.
# On Windows that fan-out is fragile and a worker can die mid-deserialize with
# "Connection reset". One partition = one worker = one deserialization pass.
# We cache() and force materialization with count() so every later action
# (join, write, show) reads cached JVM rows and never re-invokes Python.
msd_rdd = spark.sparkContext.parallelize(msd_records, numSlices=1)
msd_spark = spark.createDataFrame(msd_rdd, schema=msd_schema)
msd_spark = msd_spark.dropna(subset=MSD_AUDIO_FEATURES).cache()
print(f"Loaded {msd_spark.count():,} valid MSD tracks")

# Remove MSD tracks that already exist in our Spotify dataset.
# We match on lower-cased track_name + artist_name to catch case differences.
# left_anti join keeps only rows in msd that have NO match in spotify_keys.
spotify_keys = songs_df_finalized.select(
    lower(trim(col("track_name"))).alias("name_key"),
    lower(trim(col("artist_name"))).alias("artist_key")
)

msd_keyed = msd_spark \
    .withColumn("name_key",   lower(trim(col("track_name")))) \
    .withColumn("artist_key", lower(trim(col("artist_name"))))

msd_cleaned = msd_keyed \
    .join(spotify_keys, ["name_key", "artist_key"], "left_anti") \
    .drop("name_key", "artist_key")

print(f"MSD after removing Spotify overlaps: {msd_cleaned.count():,}")

spark_write_csv(msd_cleaned, f"{PROCESSED_DIR}/msd_cleaned")
msd_cleaned.show(5)

Found 10,000 MSD files
Loaded 10,000 valid MSD tracks
MSD after removing Spotify overlaps: 9,465
Saved → ../data/processed/msd_cleaned/
+------------------+--------------------+---------------+-----+----------+------------+------+-------+
|          track_id|          track_name|    artist_name|genre|popularity|danceability|energy|  tempo|
+------------------+--------------------+---------------+-----+----------+------------+------+-------+
|SOAUTVB12AB018AFF0|                    | Sébastien Roch| NULL|      NULL|         0.0|   0.0|126.103|
|SOCJFFQ12A8C13E02A|'Ti Monde (LP Ver...|     BeauSoleil| NULL|       0.0|         0.0|   0.0|119.663|
|SORAETJ12A58A7DAA3|(I'm So) Afraid O...|  Charley Pride| NULL|      NULL|         0.0|   0.0| 98.824|
|SOPNFYY12AB01828D4|...and Heavens Cr...|Swallow The Sun| NULL|      64.9|         0.0|   0.0| 71.758|
|SOESTLC12A58A767BC|         100% Dundee|      The Roots| NULL|      NULL|         0.0|   0.0|183.486|
+------------------+--------------------

In [14]:
# Split MSD into train/test (80/20) — same strategy as Spotify songs.
# Used in notebook 04 to evaluate cosine similarity on MSD tracks separately.
msd_train_df, msd_test_df = msd_cleaned.randomSplit([0.8, 0.2], seed=42)

spark_write_csv(msd_train_df, f"{SPLIT_DIR}/msd_train")
spark_write_csv(msd_test_df,  f"{SPLIT_DIR}/msd_test")

print(f"MSD Train : {msd_train_df.count():,} rows")
print(f"MSD Test  : {msd_test_df.count():,} rows")

Saved → ../outputs/train_test_split/msd_train/
Saved → ../outputs/train_test_split/msd_test/
MSD Train : 7,595 rows
MSD Test  : 1,870 rows


## 2.6 - Clean ALS Interaction Data (User–Song Dataset)


In [15]:
# Load the playlist dataset with Spark.
# DROPMALFORMED silently skips rows where song/playlist names contain
# unescaped commas that break the CSV parser — same effect as pandas on_bad_lines='skip'.
df_als_raw = spark.read.csv(
    f"{DATA_DIR}/spotify_playlists/spotify_dataset.csv",
    header=True,
    quote='"',
    ignoreLeadingWhiteSpace=True,
    mode="DROPMALFORMED"
)

# The raw CSV has column names with leading spaces and quote characters,
# e.g. ' "user_id"'. We strip them to get clean names: user_id, trackname, etc.
clean_cols = [c.strip().strip('"') for c in df_als_raw.columns]
df_als = df_als_raw.toDF(*clean_cols)

# Drop rows missing user, track, or artist — these are useless for ALS.
# Then remove duplicate (user, track, artist) triples — a user can only
# "interact" with a song once; duplicates don't add information.
df_als = df_als.dropna(subset=["user_id", "trackname", "artistname"])
df_als = df_als.dropDuplicates(["user_id", "trackname", "artistname"])

# Combine track name + artist into one key to uniquely identify a song
# across different playlists (same song can have slightly different IDs).
df_als = df_als.withColumn("track_key", concat_ws("||", col("trackname"), col("artistname")))

# Encode user_id as a sequential integer (0, 1, 2, ...).
# ALS in Spark MLlib requires integer IDs, not strings.
# We use zipWithIndex instead of StringIndexer to avoid OOM —
# StringIndexer collects all unique values to the driver, which crashes on 2.7M songs.
user_mapping = df_als.select("user_id").distinct() \
    .rdd.zipWithIndex() \
    .map(lambda x: (x[0][0], int(x[1]))) \
    .toDF(["user_id_original", "user_id_enc"])

# Same encoding for song track_key → integer song ID
song_mapping = df_als.select("track_key").distinct() \
    .rdd.zipWithIndex() \
    .map(lambda x: (x[0][0], int(x[1]))) \
    .toDF(["track_key_ref", "song_id_enc"])

# Join the integer encodings back to the main DataFrame
df_als = df_als.join(user_mapping, df_als["user_id"] == user_mapping["user_id_original"]) \
               .drop("user_id_original")
df_als = df_als.join(song_mapping, df_als["track_key"] == song_mapping["track_key_ref"]) \
               .drop("track_key_ref")

# Implicit feedback: since we have no star ratings, we assign 1.0 to every
# interaction — a song appearing in a playlist signals the user likes it.
df_als = df_als.withColumn("interaction", lit(1.0))

# Keep only the columns needed for ALS training + human-readable labels for debugging
users_ds_final = df_als.select(
    col("user_id_enc").alias("user_id"),
    col("song_id_enc").alias("song_id"),
    col("interaction"),
    col("user_id").alias("user_id_original"),
    col("trackname").alias("track_name"),
    col("artistname").alias("artist_name"),
    col("playlistname").alias("playlist_name")
)

print(f"Interactions : {users_ds_final.count():,}")
print(f"Unique users : {users_ds_final.select('user_id').distinct().count():,}")
print(f"Unique songs : {users_ds_final.select('song_id').distinct().count():,}")
users_ds_final.show(5)

Interactions : 11,383,850
Unique users : 15,914
Unique songs : 2,795,067
+-------+-------+-----------+--------------------+--------------------+-----------------+--------------------+
|user_id|song_id|interaction|    user_id_original|          track_name|      artist_name|       playlist_name|
+-------+-------+-----------+--------------------+--------------------+-----------------+--------------------+
|   5465|  88381|        1.0|4ca73e53b015472ce...| Daedelus - Looki...|         Daedelus|KCRW Music Mine: ...|
|   6851| 133839|        1.0|ad514347f1df0ebee...|      ! Ay Carmela ¡|Estudios Talkback|The Great Songs o...|
|   3742|  75668|        1.0|7ee2b92c5bcf6133b...|     !*?&#! Truckers|    Doug Stanhope|        Voices-Voces|
|   7069| 133890|        1.0|6c00c36a0a3983244...|            !Basura!| Doctor Explosion|                 PE2|
|  10608|  91976|        1.0|957f41183dca65713...|  !Fire In the Barn!|Pancake Breakfast|          LIVE ACSTC|
+-------+-------+-----------+----------

In [16]:
# Save the full cleaned interactions dataset
spark_write_csv(users_ds_final, f"{PROCESSED_DIR}/users_interactions_cleaned")
print(f"Total rows : {users_ds_final.count():,}")
print(f"Columns    : {users_ds_final.columns}")

Saved → ../data/processed/users_interactions_cleaned/
Total rows : 11,383,850
Columns    : ['user_id', 'song_id', 'interaction', 'user_id_original', 'track_name', 'artist_name', 'playlist_name']


## 2.7 - ALS Train/Test Split


In [17]:
# Split interactions 80/20.

users_train_ds, users_test_ds = users_ds_final.randomSplit([0.8, 0.2], seed=42)

spark_write_csv(users_train_ds, f"{SPLIT_DIR}/users_train")
spark_write_csv(users_test_ds,  f"{SPLIT_DIR}/users_test")

print(f"ALS Train : {users_train_ds.count():,} rows")
print(f"ALS Test  : {users_test_ds.count():,} rows")
print(f"Unique users in train : {users_train_ds.select('user_id').distinct().count():,}")
print(f"Unique songs in train : {users_train_ds.select('song_id').distinct().count():,}")

Saved → ../outputs/train_test_split/users_train/
Saved → ../outputs/train_test_split/users_test/
ALS Train : 9,100,076 rows
ALS Test  : 2,274,514 rows
Unique users in train : 15,849
Unique songs in train : 2,429,069


In [18]:
# Always stop the Spark session at the end to release JVM memory and ports.
# Not stopping it means the background JVM process keeps running.
spark.stop()
print("Done. Spark session closed.")

Done. Spark session closed.
